In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from keras.layers import InputLayer,Input
import random
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.metrics import R2Score

SEED = 29

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [2]:
df = pd.read_parquet("train.parquet")

In [3]:
seqs = np.zeros((517,1000,32))
for i in range(517):
   seqs[i] = df[df["seq_ix"] == i].drop(columns= ["need_prediction","seq_ix","step_in_seq"]).values

diff_seqs = np.zeros((517,999,32))
for i in range(517):
   for j in range(999):
    diff_seqs[i][j] = seqs[i][j+1] - seqs[i][j]



In [16]:
input_to_encoder_seqs = np.zeros((250,100,32))
for i in range(250):
   input_to_encoder_seqs[i] = seqs[i][0:100][:]


In [17]:
output_of_decoder_seqs = np.zeros((250,899,32))
for i in range(250):
   output_of_decoder_seqs[i] =  seqs[i][100:999][:]
   


In [9]:
import tensorflow.keras.backend as K
def r2_keras(y_true, y_pred):
    """
    Custom R2-score metric for 3D tensors (batch, time_steps, features).
    Flattens the time steps dimension before calculating R2.
    """
    # Reshape tensors from (batch, time_steps, features) to (batch * time_steps, features)
    y_true_reshaped = K.reshape(y_true, (-1, K.shape(y_true)[-1]))
    y_pred_reshaped = K.reshape(y_pred, (-1, K.shape(y_pred)[-1]))

    # Calculate Sum of Squares Residual (SS_res)
    SS_res = K.sum(K.square(y_true_reshaped - y_pred_reshaped))

    # Calculate Total Sum of Squares (SS_tot)
    # Mean is calculated across all flattened samples
    SS_tot = K.sum(K.square(y_true_reshaped - K.mean(y_true_reshaped)))

    # Calculate R2 score, add epsilon to prevent division by zero
    r2 = 1 - SS_res / (SS_tot + K.epsilon())

    return r2

In [22]:
class AutoEncoder(Model):
  def __init__(self,input_shape):
    self.num_neurons = [40,40,40]
    
    self.enc_dims = 64
    self.decoder_neurons = self.num_neurons[-1]
    self.to_predict = 899
    # self.encoder_layers = []
    # self.decoder_layers = []
    super(AutoEncoder,self).__init__()
    


    self.encoder_lstm_1 = layers.LSTM(self.num_neurons[0],
                                          return_sequences=True,
                                          name="encoder_lstm_1",
                                          recurrent_dropout = 0.3) 
    self.encoder_lstm_middle = layers.LSTM(self.num_neurons[1],
                                          return_sequences=True,
                                          name="encoder_lstm_middle")
    self.projector_layer = layers.Dense(self.num_neurons[1],activation="linear",name = "projector")
    
    self.encoder_lstm_2 = layers.LSTM(self.num_neurons[2],
                                          return_sequences=True,
                                          return_state=True,
                                          name="encoder_lstm_2")

    self.decoder_lstm = layers.LSTM(self.decoder_neurons,
                                         return_sequences=True,
                                         return_state=True,
                                         name="decoder_lstm")
    
    self.norm_1 = layers.LayerNormalization(name = "norm1")
    self.norm_2 = layers.LayerNormalization(name = "norm2")
    self.norm_3 = layers.LayerNormalization(name = "norm3")
    self.norm_4 = layers.LayerNormalization(name = "norm4")

    self.DenseQ = layers.Dense(self.enc_dims,name = "Query_Dense")
    self.DenseK = layers.Dense(self.enc_dims,name = "Query_Key")
    self.DenseV = layers.Dense(self.enc_dims,name = "Query_Value")
    self.dropout = layers.Dropout(0.3)

    # self.FFn_neurons = [24,36,self.enc_dims]

    # self.FFn_Dense1 = layers.Dense()

    num_heads = 4
    self.self_attention_layer = layers.MultiHeadAttention(name = "self_attention",num_heads = num_heads,key_dim = self.enc_dims//num_heads)
    self.cross_attention_layer = layers.MultiHeadAttention(name="attention",num_heads = num_heads,key_dim= self.enc_dims//num_heads)
    
    self.concat_layer = layers.Concatenate(axis=-1, name="concat_attention")
    self.output_dense_layer = layers.Dense(input_shape[1], name="output_dense",activation="linear") 
    self.output_time_dist = layers.TimeDistributed(self.output_dense_layer, name="output_time_distributed") 



    inputs = tf.keras.Input(input_shape,name = "init_input")  
    x1 = self.encoder_lstm_1(inputs)
    x = self.encoder_lstm_middle(x1)
    x1_proj = self.projector_layer(x1)
    x = layers.Add(name = "skip_conn")([x1_proj,x])
    x = self.norm_1(x)
    
    lstm_out , h , c = self.encoder_lstm_2(x)
    states = [h,c]

    key = self.DenseK(lstm_out)
    query = self.DenseQ(lstm_out)
    val = self.DenseV(lstm_out)

    attention_context_for_enc = self.self_attention_layer(query =query , value = val,key =key)
    encoder_out = self.concat_layer([attention_context_for_enc,lstm_out])
    encoder_out = self.norm_2(encoder_out)


    self.encoder_model = Model(inputs = inputs,outputs = [encoder_out] + states , name = "ENCODER")

    target = tf.keras.Input(shape = (self.to_predict,input_shape[1]))
    decoder_lstm_outputs, _, _ = self.decoder_lstm(target,initial_state=states)



    attention_context = self.cross_attention_layer(decoder_lstm_outputs, encoder_out,encoder_out)
    decoder_combined_context = self.concat_layer([attention_context, decoder_lstm_outputs])
    decoder_combined_context = self.norm_3(decoder_combined_context)
    decoder_combined_context = self.dropout(decoder_combined_context)
    decoder_pred_outputs = self.output_time_dist(decoder_combined_context)


    self.training_model = Model(inputs=[inputs, target],outputs=decoder_pred_outputs)

    #----------------------------------------------------------------------------------------------------
    inf_decoder_input_prev = Input(shape=(1, input_shape[1]), name="inf_decoder_input") # Previous prediction
    inf_decoder_state_h = Input(shape=(self.decoder_neurons,), name="inf_state_h")          # Previous h state
    inf_decoder_state_c = Input(shape=(self.decoder_neurons,), name="inf_state_c")          # Previous c state
    inf_decoder_states_prev = [inf_decoder_state_h, inf_decoder_state_c]                 # Combined previous states
    inf_encoder_outputs = Input(shape=(input_shape[0], self.num_neurons[-1]+self.enc_dims), name="inf_encoder_outputs") # ALL encoder outputs for attention

        # Connect layers for one step using shared layer instances:
    inf_dec_lstm_out, inf_state_h_out, inf_state_c_out = self.decoder_lstm(
        inf_decoder_input_prev, initial_state=inf_decoder_states_prev # Initialize with previous states
    )
    inf_attention_context = self.cross_attention_layer(inf_dec_lstm_out, inf_encoder_outputs,inf_encoder_outputs) # Calculate attention for this step
    inf_combined_context = self.concat_layer([inf_attention_context, inf_dec_lstm_out])    # Combine
    inf_decoder_pred = self.output_time_dist(inf_combined_context)                       # Predict (shape=(batch, 1, features))

        # Define the single-step inference decoder model
    self.inf_decoder_model = Model(
        inputs=[inf_decoder_input_prev, inf_decoder_states_prev, inf_encoder_outputs],
        outputs=[inf_decoder_pred] + [inf_state_h_out, inf_state_c_out], # Output prediction AND next states
        name="inf_decoder"
    )      

    
                            
  

  def train(self,X_train,y_train):
    last_known_step = X_train[:, -1:, :] 
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_r2_keras',  
        factor=0.5,          
        patience=2,          
        verbose=1,           
        mode='max',         
        min_delta=0.002,      
        cooldown=2,          
        min_lr=0.000002        
    )
    future_inputs = y_train[:, :-1, :]

    # learning_scheduler = tf.keras.optimizers.schedules.ExponentialDecay(
    #     0.01,
    #     decay_steps=10000,
    #     decay_rate=0.96,
    #     staircase=True)
    
    # 3. CREATE THE NEW DECODER INPUT VARIABLE
    # This is the "shifted" sequence for teacher forcing
    # Shape: (batch_size, 899, 32)
    decoder_input_data = np.concatenate([last_known_step, future_inputs], axis=1)
    
    # 4. THE TARGET IS JUST THE ORIGINAL Y_TRAIN
    decoder_target_data = y_train
    optim = tf.keras.optimizers.AdamW(learning_rate=0.01,clipnorm = 1.0)
    self.training_model.compile(optimizer = optim,loss= "mse",metrics=[r2_keras])
    self.training_model.summary()
    self.training_model.fit([X_train,decoder_input_data],
                            y_train,batch_size = 32, 
                            epochs = 25,
                            verbose = True, 
                            validation_split = 0.2,
                            callbacks = [reduce_lr]
                            )
    self.training_model.save_weights("C:/CodingWorkspace/Python/WunderRNN/AE_wts_diffs.weights.h5")
    


In [23]:
autoenc = AutoEncoder([100,32])
autoenc.train(X_train = input_to_encoder_seqs,y_train=output_of_decoder_seqs)

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ init_input          │ (None, 100, 32)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_1      │ (None, 100, 40)   │     11,680 │ init_input[0][0]  │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ projector (Dense)   │ (None, 100, 40)   │      1,640 │ encoder_lstm_1[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_middle │ (None, 100, 40)   │     12,960 │ encoder_lstm_1[0… │
│ (LSTM)              │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ skip_conn (Add)     │ (None, 100, 40)   │          0 │ projector[0][0],  │
│                     │                   │            │ encoder_lstm_mid… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ norm1               │ (None, 100, 40)   │         80 │ skip_conn[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm_2      │ [(None, 100, 40), │     12,960 │ norm1[0][0]       │
│ (LSTM)              │ (None, 40),       │            │                   │
│                     │ (None, 40)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Query_Key (Dense)   │ (None, 100, 64)   │      2,624 │ encoder_lstm_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Query_Dense (Dense) │ (None, 100, 64)   │      2,624 │ encoder_lstm_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Query_Value (Dense) │ (None, 100, 64)   │      2,624 │ encoder_lstm_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ self_attention      │ (None, 100, 64)   │     16,640 │ Query_Key[0][0],  │
│ (MultiHeadAttentio… │                   │            │ Query_Dense[0][0… │
│                     │                   │            │ Query_Value[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_attention    │ (None, 1, 80)     │          0 │ self_attention[0… │
│ (Concatenate)       │                   │            │ encoder_lstm_2[0… │
│                     │                   │            │ attention[0][0],  │
│                     │                   │            │ decoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_5       │ (None, 899, 32)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 1, 40),   │     11,680 │ input_layer_5[0]… │
│                     │ (None, 40),       │            │ encoder_lstm_2[0… │
│                     │ (None, 40)]       │            │ encoder_lstm_2[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ norm2               │ (None, 100, 104)  │        208 │ concat_attention… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 1, 40)     │     18,664 │ decoder_lstm[0][… │
│ (MultiHeadAttentio… │                   │            │ norm2[0][0],      │
│                     │                   │            │ norm2[0][0]     

 Total params: 97,136 (379.44 KB)

 Trainable params: 97,136 (379.44 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 43s 2s/step - loss: 2.0386 - r2_keras: -0.9600 - val_loss: 0.9101 - val_r2_keras: 0.0943 - learning_rate: 0.0100
Epoch 2/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 628ms/step - loss: 1.1934 - r2_keras: -0.1908 - val_loss: 0.8118 - val_r2_keras: 0.1946 - learning_rate: 0.0100
Epoch 3/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 613ms/step - loss: 0.9907 - r2_keras: 0.0134 - val_loss: 0.7592 - val_r2_keras: 0.2472 - learning_rate: 0.0100
Epoch 4/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 618ms/step - loss: 0.8608 - r2_keras: 0.1423 - val_loss: 0.7123 - val_r2_keras: 0.2960 - learning_rate: 0.0100
Epoch 5/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 631ms/step - loss: 0.7950 - r2_keras: 0.2066 - val_loss: 0.7019 - val_r2_keras: 0.3071 - learning_rate: 0.0100
Epoch 6/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 637ms/step - loss: 0.7636 - r2_keras: 0.2372 - val_loss: 0.6899 - val_r2_keras: 0.3191 - learning_rate: 0.0100
Epoch 7/25
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 628ms/step - loss: 0.7476 - r2_keras: 0.2525 - val_loss: 0.67